In [ ]:
from __future__ import annotations

import argparse
import re
from functools import reduce
from pathlib import Path

import pandas as pd


FILE_PATTERN = re.compile(r"^[^_]+_([A-Z]\d{2})_\d{3}_\d{2}\.dat$", re.IGNORECASE)


def parse_satellite_from_filename(file_path: Path) -> str:
    match = FILE_PATTERN.match(file_path.name)
    if not match:
        raise ValueError(f"Cannot parse satellite from filename: {file_path.name}")
    return match.group(1).upper()


def parse_dat_file(file_path: Path) -> pd.DataFrame:
    satellite = parse_satellite_from_filename(file_path)

    rows: list[dict[str, float | int | str]] = []
    for line in file_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue

        parts = stripped.split()
        if len(parts) < 7:
            continue

        try:
            rows.append(
                {
                    "tsn": int(parts[0]),
                    "hour": float(parts[1]),
                    "el": float(parts[2]),
                    "az": float(parts[3]),
                    "tec_l1l2": float(parts[4]),
                    "tec_c1p2": float(parts[5]),
                    "validity": int(float(parts[6])),
                    "satellite": satellite,
                }
            )
        except ValueError:
            continue

    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame

    satellite_cols = ["hour", "el", "az", "tec_l1l2", "tec_c1p2", "validity"]
    rename_map = {col: f"{col}_{satellite}" for col in satellite_cols}
    frame = frame[["tsn", *satellite_cols]].rename(columns=rename_map)
    return frame


def parse_station_folder(folder_path: Path) -> pd.DataFrame:
    dat_files = sorted(folder_path.glob("*.dat"))
    if not dat_files:
        raise FileNotFoundError(f"No .dat files found in: {folder_path}")

    frames = [parse_dat_file(path) for path in dat_files]
    frames = [frame for frame in frames if not frame.empty]
    if not frames:
        raise ValueError(f"No parseable data rows found in: {folder_path}")

    merged = reduce(lambda left, right: pd.merge(left, right, on="tsn", how="outer"), frames)
    merged = merged.sort_values("tsn").reset_index(drop=True)
    return merged


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Parse all station .dat files into one DataFrame"
    )
    parser.add_argument(
        "--folder",
        default=r"N:\abstec-suite\in\2026\001\ozer0010",
        help="Folder with per-satellite .dat files",
    )
    parser.add_argument(
        "--output-csv",
        help="Optional output CSV path",
    )
    return parser.parse_args()


def main() -> None:
    folder_path = Path(r"N:\abstec-suite\in\2026\001\ozer0010")
    output_csv = r"N:\abstec-suite\experiments\combined_data_merged_by_tsn.csv"
    if not folder_path.exists():
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    dataframe = parse_station_folder(folder_path)

    print(f"Rows: {len(dataframe)}")
    print(f"Columns: {len(dataframe.columns)}")
    print(dataframe.head(10).to_string(index=False))

    if output_csv:
        output_path = Path(output_csv)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        dataframe.to_csv(output_path, index=False)
        print(f"Saved CSV: {output_path.resolve()}")


if __name__ == "__main__":
    main()


Rows: 2799
Columns: 337
 tsn  hour_G01   el_G01    az_G01  tec_l1l2_G01  tec_c1p2_G01  validity_G01  hour_G02   el_G02    az_G02  tec_l1l2_G02  tec_c1p2_G02  validity_G02  hour_G03   el_G03    az_G03  tec_l1l2_G03  tec_c1p2_G03  validity_G03  hour_G04   el_G04    az_G04  tec_l1l2_G04  tec_c1p2_G04  validity_G04  hour_G05  el_G05  az_G05  tec_l1l2_G05  tec_c1p2_G05  validity_G05  hour_G06  el_G06  az_G06  tec_l1l2_G06  tec_c1p2_G06  validity_G06  hour_G07  el_G07  az_G07  tec_l1l2_G07  tec_c1p2_G07  validity_G07  hour_G08  el_G08  az_G08  tec_l1l2_G08  tec_c1p2_G08  validity_G08  hour_G09  el_G09  az_G09  tec_l1l2_G09  tec_c1p2_G09  validity_G09  hour_G10  el_G10  az_G10  tec_l1l2_G10  tec_c1p2_G10  validity_G10  hour_G11  el_G11  az_G11  tec_l1l2_G11  tec_c1p2_G11  validity_G11  hour_G12  el_G12   az_G12  tec_l1l2_G12  tec_c1p2_G12  validity_G12  hour_G13  el_G13  az_G13  tec_l1l2_G13  tec_c1p2_G13  validity_G13  hour_G14  el_G14  az_G14  tec_l1l2_G14  tec_c1p2_G14  validity_G14  hour_

: 